# 4.1.7 Průzkumná analýza dat

Jednoduchý notebook pro základní průzkumnou analýzu dat před nahráním do databáze.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

for _ in range(5):
    if (PROJECT_ROOT / "docs").exists() and (PROJECT_ROOT / "data").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


PROJECT_ROOT: /Users/adelaleppeltova/firesport-app


## Načtení dat


In [2]:
%pip install pandas



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

from docs.analysis import (
    add_quality_flags,
    find_inconsistent_team_names,
    find_missing_attempts,
    find_possible_duplicate_athletes,
    get_competitions,
    get_results,
    load_json_files_from_dir,
    results_to_dataframe,
    summarize_data_quality,
    summarize_dataset,
    summarize_final_times,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


In [4]:
# Upravte cestu podle umístění vašich JSON souborů.
DATA_DIR = PROJECT_ROOT / "data"

records = load_json_files_from_dir(DATA_DIR)
competitions = get_competitions(records)
results = get_results(records)

df_competitions = pd.DataFrame(competitions)
df_results = add_quality_flags(results_to_dataframe(results))

len(records), len(df_competitions), len(df_results)


(329, 329, 13854)

## Charakteristika datasetu


In [5]:
dataset_summary = summarize_dataset(df_results)
pd.Series(dataset_summary)


athlete_count              4708
result_count              13854
competition_count            65
min_date             2019-05-04
max_date             2025-10-04
disciplines              [100m]
dtype: object

In [6]:
df_results[[
    "competition_date",
    "competition_name",
    "category_name",
    "discipline",
    "athlete_full_name",
    "team",
    "final_time",
    "final_status",
]].head(10)


,competition_date,competition_name,category_name,discipline,athlete_full_name,team,final_time,final_status
0,2019-06-02,Běloveský kilo,Muži,100m,Patrik Kligl,Běloves,15.43,valid
1,2019-06-02,Běloveský kilo,Muži,100m,Lukáš Kroupa,Kvasiny B,16.01,valid
2,2019-06-02,Běloveský kilo,Muži,100m,Jan Zhříval,HZS Hradec Králové,16.13,valid
3,2019-06-02,Běloveský kilo,Muži,100m,Michal Gierlowski,Tuř,17.34,valid
4,2019-06-02,Běloveský kilo,Muži,100m,Pavel Hoffman,Běloves,17.52,valid
5,2019-06-02,Běloveský kilo,Muži,100m,David Hoffman,Běloves,17.59,valid
6,2019-06-02,Běloveský kilo,Muži,100m,Dominik Mašek,Ruda,17.68,valid
7,2019-06-02,Běloveský kilo,Muži,100m,Jakub Vondra,Stolín,18.19,valid
8,2019-06-02,Běloveský kilo,Muži,100m,Stanislav Pavlas,Vidochov,18.63,valid
9,2019-06-02,Běloveský kilo,Muži,100m,Lukáš Kligl,Běloves,18.72,valid


## Základní statistické ukazatele


In [7]:
final_time_summary = summarize_final_times(df_results)
pd.Series(final_time_summary)


mean      20.268042
median    19.720000
min       14.640000
max       57.130000
std        2.916190
dtype: float64

## Kvalita dat


In [8]:
quality_summary = summarize_data_quality(df_results)
pd.Series(quality_summary)


missing_birth_year_count       6233
missing_fscode_count           6155
missing_team_count                2
missing_attempts_count            0
suspicious_final_time_count     524
duplicate_rows_count              0
dtype: int64

In [9]:
df_results[[
    "athlete_full_name",
    "team",
    "fscode",
    "birth_year",
    "final_time",
    "final_status",
    "has_missing_attempts",
    "suspicious_final_time",
]].head(10)


,athlete_full_name,team,fscode,birth_year,final_time,final_status,has_missing_attempts,suspicious_final_time
0,Patrik Kligl,Běloves,NaN,NaN,15.43,valid,False,False
1,Lukáš Kroupa,Kvasiny B,NaN,NaN,16.01,valid,False,False
2,Jan Zhříval,HZS Hradec Králové,NaN,NaN,16.13,valid,False,False
3,Michal Gierlowski,Tuř,NaN,NaN,17.34,valid,False,False
4,Pavel Hoffman,Běloves,NaN,NaN,17.52,valid,False,False
5,David Hoffman,Běloves,NaN,NaN,17.59,valid,False,False
6,Dominik Mašek,Ruda,NaN,NaN,17.68,valid,False,False
7,Jakub Vondra,Stolín,NaN,NaN,18.19,valid,False,False
8,Stanislav Pavlas,Vidochov,NaN,NaN,18.63,valid,False,False
9,Lukáš Kligl,Běloves,NaN,NaN,18.72,valid,False,False


## Identifikace problémů v datech


In [10]:
possible_duplicate_athletes = find_possible_duplicate_athletes(df_results)
possible_duplicate_athletes.head(20)


,athlete_full_name,team_count,fscode_count,birth_year_count,teams,fs_codes,birth_years,result_count
0,Lukáš Kroupa,6,1,1,"[Kvasiny B, Pardubice - Polabiny, Pardubice- Polabiny, Pardubice-Polabiny, VHJ Pardubice, Zbožnov]",[17681.0],[1997.0],35
1,Šimon Šuba,3,1,1,"[Císařov, Milotice nad Bečvou, Oznice]",[29131.0],[2006.0],33
2,Jakub Mikulík,4,1,1,"[Bludov, Býškovice, Hostinné, SLATINY, Slatiny]",[49721.0],[2004.0],31
3,Matyáš Novák,4,2,2,"[Krouna, Seč, Stará Říše, Výčapy]","[50901.0, 94551.0]","[2006.0, 2007.0]",31
4,Vít Vymazal,4,1,1,"[Morkovice, Oznice, SLATINY, Seč, Slatiny]",[28381.0],[2006.0],30
5,Anna Rajnetová,3,1,1,"[Pardubice Polabiny, Pardubice-Polabiny, Pardubice–Polabiny]",[46242.0],[2007.0],29
6,František Rajnet,4,1,1,"[Pardubice - Polabiny, Pardubice Polabiny, Pardubice-Polabiny, Stéblová]",[61611.0],[2007.0],29
7,Tereza Kroupová,2,1,1,"[CHÁBORY, Chábory, Kvasiny]",[10732.0],[2002.0],29
8,Simona Páralová,1,1,2,[Bořitov],[26472.0],"[2003.0, 2004.0]",27
9,Dominik Andrée,4,1,1,"[Bludov, Kly, Lhenice, PS Mělník]",[48121.0],[2003.0],26


In [11]:
inconsistent_team_names = find_inconsistent_team_names(df_results)
inconsistent_team_names.head(20)


,team_key,variant_count,variants,result_count
0,bukovice,2,"[BUKOVICE, Bukovice]",322
1,slatiny,2,"[SLATINY, Slatiny]",148
2,hzs kraje vysočina,2,"[HZS Kraje Vysočina, HZS kraje Vysočina]",68
3,chábory,2,"[CHÁBORY, Chábory]",58
4,opočno,2,"[OPOČNO, Opočno]",50
5,dubenec,2,"[DUBENEC, Dubenec]",46
6,libňatov,2,"[LIBŇATOV, Libňatov]",39
7,kostelec nad černými lesy,2,"[Kostelec nad Černými Lesy, Kostelec nad Černými lesy]",32
8,dobrá voda,2,"[DOBRÁ VODA, Dobrá Voda]",29
9,havířov-město,2,"[Havířov-Město, Havířov-město]",27


In [13]:
missing_attempts = find_missing_attempts(df_results)

print(f"Pocet zaznamu s chybejicimi nebo nekompletnimi pokusy: {len(missing_attempts)}")

missing_attempts[[
    "competition_date",
    "discipline",
    "athlete_full_name",
    "team",
    "times",
    "has_missing_attempts",
]].head(20)

Pocet zaznamu s chybejicimi nebo nekompletnimi pokusy: 0


,competition_date,discipline,athlete_full_name,team,times,has_missing_attempts
